# 03 — Siam-GCAN (Siamese Graph Convolutional Attention Network)

**Family:** Graph CNN + Siamese &nbsp;•&nbsp; **Input:** DE feature pairs `[14 x 5]`

A twin-branch, weight-shared GCN encoder with multi-head self-attention over the
electrode graph. Training uses **trial pairs**: the contrastive term pulls
same-emotion windows together and pushes different-emotion windows apart in the
embedding space, on top of a per-branch cross-entropy term. Inference uses a
single branch, so the shared evaluation harness works unchanged.

## 1. Setup

In [ ]:
import os, sys, time, json
from functools import partial

import numpy as np
import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.evaluation import (
    set_seed, get_device, count_parameters,
    make_default_fit, default_predict,
    run_subject_dependent, run_loso, save_results, quick_report,
)
from src.data_loader import (
    build_subject_dependent_loaders, build_loso_loaders, load_subject, REDUCED_5_CHANNELS,
)

set_seed(42)
DEVICE = get_device()

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
SD_SPLITS     = os.path.join(ROOT, "data", "splits", "subject_dependent_splits.json")
LOSO_SPLITS   = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR   = os.path.join(ROOT, "results")

# ----- experiment configuration -----
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> a 3-subject / 3-fold smoke test with few epochs
                      # False -> the full benchmark used for the thesis table
print("device:", DEVICE, "| label:", LABEL, "| QUICK:", QUICK)

import torch.nn as nn
from torch.utils.data import DataLoader
from src.models import siam_gcan
from src.models.siam_gcan import contrastive_loss
from src.data_loader import SiamesePairDataset
MODEL_NAME = "SiamGCAN"

## 2. Architecture

`src/models/siam_gcan.py`: `GCANEncoder` = 2 × (`GraphAttnLayer` = learnable-adjacency
GCN propagation + `nn.MultiheadAttention` over electrodes + residual LayerNorm),
mean-pooled over nodes and projected to a 64-D embedding. A small MLP head maps the
embedding to class logits.

**Pair sampling** (`SiamesePairDataset`) draws, for each window, a partner that is
same-class ~50% of the time; `pair_label = 1` iff the two share an emotion class.
Pairs are resampled every epoch.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, 5)}
_m = siam_gcan.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## 3. Custom training step

The shared `make_default_fit` cannot express the paired objective, so we define a
`fit_fn` with the same signature `(model, train_loader, val_loader, device) -> seconds`.
It wraps the train loader's `EEGWindowDataset` in a `SiamesePairDataset`, then
optimizes `CE(a) + CE(b) + lambda * contrastive(emb_a, emb_b, pair_label)`.

In [ ]:
def make_siam_fit(epochs, lr, lam=1.0, pairs_per_sample=2, wd=1e-4):
    def fit(model, train_loader, val_loader, device):
        model.to(device)
        base = train_loader.dataset
        bs = train_loader.batch_size or 64
        pair_ds = SiamesePairDataset(base, pairs_per_sample=pairs_per_sample, seed=42)
        pl = DataLoader(pair_ds, batch_size=bs, shuffle=True)
        opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
        ce = nn.CrossEntropyLoss()
        t0 = time.time()
        for ep in range(epochs):
            pair_ds.resample()
            model.train()
            for xa, xb, plab, ya, yb in pl:
                xa, xb = xa.to(device), xb.to(device)
                plab, ya, yb = plab.to(device), ya.to(device), yb.to(device)
                opt.zero_grad()
                ea, eb, la, lb = model.forward_pair(xa, xb)
                loss = ce(la, ya) + ce(lb, yb) + lam * contrastive_loss(ea, eb, plab)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()
        return time.time() - t0
    return fit

## Validation protocols

Both protocols are run through the shared harness in `src/evaluation.py` so the
numbers are directly comparable across all eight notebooks.

* **Subject-dependent** — for each of the 23 subjects a fresh model is trained on
  that subject's training split and tested on their held-out split
  (`data/splits/subject_dependent_splits.json`; whole film clips are kept on one
  side of the split to avoid window leakage). We report the mean ± std of the
  per-subject scores.
* **Subject-independent / LOSO** — 23 leave-one-subject-out folds
  (`data/splits/loso_splits.json`). Fold *k* trains on 22 subjects and tests on
  the 23rd; a 10% slice of the training subjects is used for early stopping.
  We report the mean ± std across the 23 folds.

The gap between the two is the central object of study in the thesis.

### 4. Subject-dependent protocol

In [ ]:
FEATURE, CHANNELS = "de", None
BATCH_SD, EPOCHS_SD, LR = 64, 60, 1e-3

sd_loader = partial(
    build_subject_dependent_loaders,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_SD, channels=CHANNELS, normalize=True,
)
sd_subjects = ["subject_01", "subject_02", "subject_03"] if QUICK else None

sd_res = run_subject_dependent(
    build_model=siam_gcan.build,
    processed_dir=PROCESSED_DIR, splits_path=SD_SPLITS,
    loader_factory=sd_loader,
    fit_fn=make_siam_fit(epochs=(5 if QUICK else EPOCHS_SD), lr=LR),
    predict_fn=default_predict, device=DEVICE,
    model_kwargs={"n_classes": 2}, subjects=sd_subjects,
)
quick_report(MODEL_NAME, sd_res)

### 5. Subject-independent (LOSO) protocol

In [ ]:
BATCH_LOSO, EPOCHS_LOSO = 128, 45

loso_loader = partial(
    build_loso_loaders,
    processed_dir=PROCESSED_DIR, loso_path=LOSO_SPLITS,
    label=LABEL, feature=FEATURE, batch_size=BATCH_LOSO, channels=CHANNELS,
    normalize=True, val_frac=0.0,
)
loso_folds = list(range(3)) if QUICK else None

loso_res = run_loso(
    build_model=siam_gcan.build, loso_path=LOSO_SPLITS, loader_factory=loso_loader,
    fit_fn=make_siam_fit(epochs=(5 if QUICK else EPOCHS_LOSO), lr=LR),
    predict_fn=default_predict, device=DEVICE,
    model_kwargs={"n_classes": 2}, folds=loso_folds,
)
quick_report(MODEL_NAME, loso_res)

## 6. Save results

In [ ]:
save_results(
    MODEL_NAME, subject_dependent=sd_res, loso=loso_res,
    family="Graph CNN + Siamese", results_dir=RESULTS_DIR,
    config={"label": LABEL, "feature": FEATURE, "batch_sd": BATCH_SD,
            "batch_loso": BATCH_LOSO, "epochs_sd": EPOCHS_SD,
            "epochs_loso": EPOCHS_LOSO, "lr": LR, "quick": QUICK,
            "pairs_per_sample": 2, "contrastive_lambda": 1.0},
)

## Notes for the methodology chapter

* Every metric here comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* Parameter count and wall-clock training time are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` at the top to smoke-test the whole flow in a few minutes
  before committing to the full multi-hour run.
* `LABEL` switches the target between valence / arousal / dominance without any
  other change.